# Customer Segmentation Using K-Means

## Objective
Segment customers based on **Annual Income** and **Spending Score** using K-Means clustering.

**Note:** PCA is not used because dimensionality reduction is unnecessary for this small feature set.

## 1. Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style='whitegrid')
%matplotlib inline


## 2. Load and Inspect the Dataset


In [ ]:
df = pd.read_csv('../Mall_Customers.csv')

print('Shape:', df.shape)
display(df.head())
print('\nData types:')
df.info()


In [ ]:
print('Missing values:\n', df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nSummary statistics:')
display(df.describe())


## 3. Exploratory Data Analysis (EDA)

We create graphs to understand customer demographics, income, spending behavior, and relationships between variables.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['Age'], bins=15, kde=True)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Number of Customers')
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x='Gender')
plt.title('Customer Distribution by Gender')
plt.xlabel('Gender')
plt.ylabel('Number of Customers')
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['Annual Income (k$)'], bins=15, kde=True)
plt.title('Annual Income Distribution')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Number of Customers')
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['Spending Score (1-100)'], bins=15, kde=True)
plt.title('Spending Score Distribution')
plt.xlabel('Spending Score (1-100)')
plt.ylabel('Number of Customers')
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', s=80)
plt.title('Annual Income vs Spending Score')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x='Age', y='Spending Score (1-100)', ax=axes[0], s=60)
axes[0].set_title('Age vs Spending Score')
sns.scatterplot(data=df, x='Age', y='Annual Income (k$)', ax=axes[1], s=60)
axes[1].set_title('Age vs Annual Income')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()


## 4. Feature Selection

For the main clustering model, use **Annual Income** and **Spending Score**. CustomerID is only an identifier, and Age/Gender are retained for later segment profiling.

In [ ]:
features = ['Annual Income (k$)', 'Spending Score (1-100)']
X = df[features].copy()
X.head()


## 5. Feature Scaling

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled[:5]


## 6. Find the Optimal Number of Clusters

We use both the **Elbow Method** and **Silhouette Score** instead of assuming a cluster count.

In [ ]:
inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    inertias.append(model.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(K_range), inertias, marker='o')
plt.title('Elbow Method')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.xticks(list(K_range))
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(K_range), silhouette_scores, marker='o')
plt.title('Silhouette Score by Number of Clusters')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.xticks(list(K_range))
plt.show()

for k, score in zip(K_range, silhouette_scores):
    print(f'k={k}: silhouette score={score:.3f}')


In [ ]:
best_k = list(K_range)[int(np.argmax(silhouette_scores))]
print('Best k according to silhouette score:', best_k)


## 7. Train K-Means and Assign Customer Segments

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

df.head()


## 8. Visualize the Customer Segments

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', hue='Cluster', palette='tab10', s=90)

centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(centers_original[:, 0], centers_original[:, 1], marker='X', s=250, c='black', label='Centroids')

plt.title('Customer Segments: Annual Income vs Spending Score')
plt.legend(title='Cluster')
plt.show()


## 9. Profile the Customer Segments

In [ ]:
segment_profile = df.groupby('Cluster').agg(
    Customers=('CustomerID', 'count'),
    Average_Age=('Age', 'mean'),
    Average_Income=('Annual Income (k$)', 'mean'),
    Average_Spending=('Spending Score (1-100)', 'mean')
).round(2)

segment_profile


In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(x=segment_profile.index, y=segment_profile['Customers'])
plt.title('Customers per Segment')
plt.xlabel('Cluster')
plt.ylabel('Number of Customers')
plt.show()


## 10. Business Interpretation

Use the segment profile above to assign meaningful names based on actual income and spending behavior. Examples include high-value customers, potential customers, budget-conscious customers, and low-engagement customers. Do not assign names until the computed cluster averages are reviewed.

In [ ]:
print('Segment profile:')
display(segment_profile)

print('Customer counts:')
print(df['Cluster'].value_counts().sort_index())


## Conclusion

K-Means groups customers with similar income and spending behavior. The resulting segments can support targeted marketing, loyalty programs, personalized offers, and customer-retention strategies.